# Face Emotion (Vision) Training

Train the 7-class face emotion classifier used by the vision module. Artifacts are saved under models/vision/face_emotion/.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


## Data requirements

Expected dataset layout: data/raw/vision/face_emotion/train/<class>/ and data/raw/vision/face_emotion/val/<class>/. If you only have a single folder, the trainer will auto-split.


## Dataset inventory


In [ ]:
from pathlib import Path

data_root = repo_root / 'data' / 'raw' / 'vision' / 'face_emotion'
print('Face emotion root:', data_root, 'exists=', data_root.exists())

def _count_images(path):
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})

train_dir = data_root / 'train'
val_dir = data_root / 'val'
test_dir = data_root / 'test'
print('train images:', _count_images(train_dir))
print('val images:', _count_images(val_dir))
print('test images:', _count_images(test_dir))


## Links to code


- Training script: `src/train/train_face_emotion.py`


- Inference module: `app/models/vision/face_emotion_predict.py`


- API endpoint: `POST /api/vision/face_emotion` (multimodal chat)


In [ ]:
# Update paths if your data lives elsewhere.



In [ ]:
data_dir = repo_root / 'data' / 'raw' / 'vision' / 'face_emotion'
print('Data dir exists:', data_dir.exists(), data_dir)


## Train the model


In [ ]:
# You can adjust epochs, image size, and architecture.
# Default arch: small_cnn. For higher accuracy, try resnet18 with --pretrained.

# !python -m src.train.train_face_emotion --data-dir data/raw/vision/face_emotion --epochs 10 --arch resnet18 --pretrained --image-size 160 --batch-size 32 --lr 3e-4 --device auto


## Review metrics


In [ ]:
import json
metrics_path = repo_root / 'models' / 'vision' / 'face_emotion' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('best_val_acc:', metrics.get('best_val_acc'))
else:
    print('metrics.json not found yet.')


## Visuals and metrics


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

metrics_path = repo_root / 'models' / 'vision' / 'face_emotion' / 'metrics.json'
if not metrics_path.exists():
    print('metrics.json not found:', metrics_path)
else:
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    history = metrics.get('history', [])
    if not history:
        print('No history in metrics.json')
    else:
        epochs = [int(h.get('epoch', i + 1)) for i, h in enumerate(history)]
        train_loss = [h.get('train_loss') for h in history]
        val_loss = [h.get('val_loss') for h in history]
        train_acc = [h.get('train_acc') for h in history]
        val_acc = [h.get('val_acc') for h in history]
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].plot(epochs, train_loss, label='train_loss')
        axes[0].plot(epochs, val_loss, label='val_loss')
        axes[0].set_title('Loss')
        axes[0].legend()
        axes[1].plot(epochs, train_acc, label='train_acc')
        axes[1].plot(epochs, val_acc, label='val_acc')
        axes[1].set_title('Accuracy')
        axes[1].legend()
        plt.tight_layout()
        plt.show()


## Sample predictions


In [ ]:
from app.models.vision.face_emotion_predict import predict_face_emotion
from PIL import Image
import matplotlib.pyplot as plt

val_root = repo_root / 'data' / 'raw' / 'vision' / 'face_emotion' / 'val'
if not val_root.exists():
    val_root = repo_root / 'data' / 'raw' / 'vision' / 'face_emotion' / 'test'
if not val_root.exists():
    val_root = repo_root / 'data' / 'raw' / 'vision' / 'face_emotion'

samples = []
if val_root.exists():
    for p in val_root.rglob('*.jpg'):
        samples.append(p)
        if len(samples) >= 4:
            break

if not samples:
    print('No sample images found under', val_root)
else:
    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, samples):
        pred = predict_face_emotion(image_bytes=img_path.read_bytes(), filename=img_path.name)
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.set_title(str(pred.get('emotion')) + ' (' + str(pred.get('confidence')) + ')')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
